# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IbtissamToure/FlyRank-Internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

I chose Logistic Regression because this is a binary classification task: identifying content items with an observed downward trend.

It provides a simple and interpretable starting point. The target is derived from trend_direction, but trend_direction and trend_pct are excluded from the model features to prevent target leakage.

This model identifies observed decline patterns in the current snapshot; it does not predict future decline.

In [7]:
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import precision_score

# Load dataset
current = Path.cwd()
project_root = next(
    (p for p in [current, *current.parents]
     if (p / "data/raw/content_refresh_anonymized.csv").exists()),
    None
)

if project_root is None:
    raise FileNotFoundError("Dataset not found")

df = pd.read_csv(
    project_root / "data/raw/content_refresh_anonymized.csv"
)

# Define the observed decline target
df = df[
    df["trend_direction"].isin(["down", "stable", "up", "flat"])
].copy()

y = (df["trend_direction"] == "down").astype(int)

# Use only non-performance and non-label-derived features
numeric_features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "content_age_days",
    "days_since_last_update",
]

categorical_features = [
    "competition_level",
    "content_type",
    "main_intent",
    "provider_used",
    "model_used",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "char_count_tier",
]

feature_columns = numeric_features + categorical_features
X = df[feature_columns].copy()
groups = df["client_id"]

# Preprocess numeric and categorical features
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features),
])

# Logistic Regression model
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    )),
])

print("Rows used:", len(X))
print("Features:", len(feature_columns))
print("Decline cases:", int(y.sum()))
print("Decline rate:", round(y.mean() * 100, 2), "%")

Rows used: 27764
Features: 16
Decline cases: 16262
Decline rate: 58.57 %


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

I use a grouped train/test split by client_id to reduce the risk of client-specific patterns appearing in both sets.

The split keeps each client's content items together, providing a more honest test of performance on unseen clients. The random seed is fixed at 42 for reproducibility.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Split by client to prevent client overlap
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Training clients:", groups.iloc[train_idx].nunique())
print("Testing clients:", groups.iloc[test_idx].nunique())
print("Client overlap:", len(
    set(groups.iloc[train_idx]) & set(groups.iloc[test_idx])
))
print("Training decline rate:", round(y_train.mean() * 100, 2), "%")
print("Testing decline rate:", round(y_test.mean() * 100, 2), "%")

assert len(set(groups.iloc[train_idx]) & set(groups.iloc[test_idx])) == 0

Training rows: 22172
Testing rows: 5592
Training clients: 24
Testing clients: 7
Client overlap: 0
Training decline rate: 59.16 %
Testing decline rate: 56.24 %


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

I train Logistic Regression using the grouped training split only.

The comparison uses the same held-out test pages and the same metric (Precision@50). Both rankings are evaluated on pages meeting the Week 4 baseline eligibility rule of at least 3,000 impressions in 90 days.

The base rate is included to show how much better each ranking performs than selecting pages without ranking.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Train the model
model.fit(X_train, y_train)

# Prepare the same eligible test population for both rankings
evaluation = pd.DataFrame(index=X_test.index)

evaluation["actual"] = y_test
evaluation["baseline_score"] = df.loc[X_test.index, "impressions_90d"]
evaluation["model_score"] = model.predict_proba(X_test)[:, 1]

# Apply the Week 4 baseline eligibility rule
evaluation = evaluation[
    evaluation["baseline_score"] >= 3000
].copy()

if len(evaluation) < 50:
    raise ValueError("Fewer than 50 eligible test pages.")

k = 50

# Calculate Precision@50 for each ranking
baseline_top = evaluation.nlargest(k, "baseline_score")
model_top = evaluation.nlargest(k, "model_score")

comparison = pd.DataFrame([
    {
        "method": "Baseline (impressions_90d)",
        "precision_at_50": baseline_top["actual"].mean()
    },
    {
        "method": "Logistic Regression",
        "precision_at_50": model_top["actual"].mean()
    },
    {
        "method": "Base rate",
        "precision_at_50": evaluation["actual"].mean()
    }
])

print("Eligible test pages:", len(evaluation))
print("Evaluation metric: Precision@50")
display(comparison.round(3))

Eligible test pages: 1315
Evaluation metric: Precision@50


,method,precision_at_50
0,Baseline (impressions_90d),0.440
1,Logistic Regression,0.600
2,Base rate,0.434


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

The model achieved higher Precision@50 than the Week 4 baseline on the same eligible test population.

However, ranking performance alone is not enough. I will inspect false positives and permutation importance to understand where the model makes mistakes and which features influence its predictions.

The results are directional and should not be interpreted as proof of causation or future performance.

In [10]:
from sklearn.inspection import permutation_importance

# 1. Feature importance
importance = permutation_importance(
    model,
    X_test.loc[evaluation.index],
    y_test.loc[evaluation.index],
    scoring="roc_auc",
    n_repeats=5,
    random_state=42,
    n_jobs=-1
)

importance_table = pd.DataFrame({
    "feature": feature_columns,
    "importance_mean": importance.importances_mean,
    "importance_std": importance.importances_std,
}).sort_values("importance_mean", ascending=False)

print("Top 3 features:")
display(importance_table.head(3).round(4))

# 2. Inspect false positives in the model's top 50
false_positives = model_top[model_top["actual"] == 0].copy()

print("\nFalse positives in model's top 50:", len(false_positives))

details = df.loc[
    false_positives.index,
    ["content_age_days", "days_since_last_update", "word_count"]
].copy()

details["model_score"] = false_positives["model_score"]
details["baseline_score"] = false_positives["baseline_score"]
details["actual"] = false_positives["actual"]

display(details.head(3).round(3))

print("\nRandom seed: 42")
print("Model: Logistic Regression")
print("Importance method: Permutation importance")

Top 3 features:


,feature,importance_mean,importance_std
12,age_tier,0.0467,0.0055
11,model_used,0.0015,0.0068
2,cpc,0.0003,0.0008



False positives in model's top 50: 20


,content_age_days,days_since_last_update,word_count,model_score,baseline_score,actual
4255,147,20,2854.0,0.734,4668,0
15566,326,20,1750.0,0.688,8402,0
19287,280,104,1894.0,0.628,3430,0



Random seed: 42
Model: Logistic Regression
Importance method: Permutation importance


The model achieved a Precision@50 of 0.600, compared with 0.440 for the Week 4 baseline and a base rate of 0.434.

Permutation importance showed that age_tier was the most influential feature, while model_used and cpc had much smaller contributions.

The model produced 20 false positives among its top 50 predictions. Inspecting examples showed that content metadata alone does not fully explain the observed trend.

These results describe patterns in the current snapshot and do not establish causation or predict future decline.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.